# Paper finder

Describe an ML problem in plain English and get back recent research papers that could help you solve it.

**How it works**
1. **Query expansion.** Your description is turned into short academic search queries. Claude does this if an API key is set; otherwise the queries you write in the config cell are used. Queries come in two kinds:
   - `same_domain`: papers about your exact problem (e.g. tenant screening)
   - `analogous`: papers solving the same *kind* of ML problem in another field (e.g. credit default prediction)
2. **Candidate retrieval.** Each query is run against [OpenAlex](https://openalex.org) with a recency filter, and the results are pooled and deduplicated.
3. **Semantic re-ranking.** Your description and each paper's title and abstract are embedded with [SPECTER2](https://huggingface.co/allenai/specter2), a SciBERT-based model trained for scientific paper similarity. Papers are then ranked by cosine similarity.

## 1. Configure

In [ ]:
PROBLEM = """
I want to create an ML algorithm that classifies whether a tenant will be a bad tenant
or a good tenant (e.g. late rent, default, eviction) based on their application data.
"""

SINCE_YEAR = 2021        # only papers published in or after this year
PER_QUERY = 200          # OpenAlex results pulled per search query (max 200, one full page)
TOP_K = 25               # how many papers to show
CITATION_WEIGHT = 0.15   # 0 = rank purely by similarity; higher = favor well-cited papers

# Used when no ANTHROPIC_API_KEY is set (or set USE_LLM = False to always use these).
USE_LLM = True
MANUAL_QUERIES = {
    "same_domain": [
        "tenant screening machine learning",
        "rental payment default prediction",
        "eviction risk prediction",
    ],
    "analogous": [
        "credit scoring machine learning",
        "loan default prediction",
        "fairness credit risk models",
    ],
}

### API keys (both optional)
- `ANTHROPIC_API_KEY`: enables automatic query expansion with Claude. Get a key at [console.anthropic.com](https://console.anthropic.com). API billing is separate from a Claude Pro subscription.
- `OPENALEX_API_KEY`: free at [openalex.org](https://openalex.org/rest-api). Without one, OpenAlex heavily rate-limits anonymous searches and the notebook will pause and retry.

Put them in a `.env` file next to this notebook, one per line (`OPENALEX_API_KEY=...`). `.env` is gitignored, so keys stay out of the notebook and out of git. Variables already set in your shell take precedence.

In [ ]:
import os
from pathlib import Path

# Load KEY=value lines from .env (gitignored) without overriding the shell environment.
env_file = Path(".env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"\''))

print("Anthropic key:", "set" if os.environ.get("ANTHROPIC_API_KEY") else "not set")
print("OpenAlex key: ", "set" if os.environ.get("OPENALEX_API_KEY") else "not set")

## 2. Query expansion (when Claude API is available)

In [ ]:
import json

EXPANSION_SCHEMA = {
    "type": "object",
    "properties": {
        "same_domain": {"type": "array", "items": {"type": "string"}},
        "analogous": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["same_domain", "analogous"],
    "additionalProperties": False,
}

EXPANSION_PROMPT = """I'm searching the academic literature (via the OpenAlex keyword search API) for recent papers that would help me build this machine learning system:

<problem>
{problem}
</problem>

Write short keyword search queries (2-5 words each, the way researchers would phrase titles and abstracts, not how a layperson would):
- same_domain: 4-6 queries for papers about this exact application, using the terminology and synonyms researchers in that field use.
- analogous: 4-6 queries for papers that solve the same kind of ML problem in a different domain, where methods, datasets, or lessons would transfer. Include an angle on fairness, bias, or regulation if the application affects people.

Avoid overly generic queries like "machine learning" on their own; each query should narrow to a useful set of papers."""


def expand_with_claude(problem: str) -> dict:
    import anthropic
    client = anthropic.Anthropic()
    response = client.beta.messages.create(
        model="claude-opus-5",
        max_tokens=2000,
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": EXPANSION_SCHEMA}},
        # If the request is ever declined by a safety classifier, retry on a fallback model.
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=[{"role": "user", "content": EXPANSION_PROMPT.format(problem=problem.strip())}],
    )
    if response.stop_reason == "refusal":
        raise RuntimeError("Claude declined the query-expansion request")
    text = next(b.text for b in response.content if b.type == "text")
    return json.loads(text)


if USE_LLM and os.environ.get("ANTHROPIC_API_KEY"):
    try:
        queries = expand_with_claude(PROBLEM)
        print("Queries generated by Claude:")
    except Exception as e:
        print(f"Claude expansion failed ({type(e).__name__}: {e}); falling back to MANUAL_QUERIES.")
        queries = MANUAL_QUERIES
else:
    queries = MANUAL_QUERIES
    print("Using MANUAL_QUERIES:")

for kind, qs in queries.items():
    print(f"  {kind}:")
    for q in qs:
        print(f"    - {q}")

## 3. Retrieve candidates from OpenAlex

In [ ]:
import time
import requests
from tqdm.auto import tqdm
from paper import Paper  # paper.py, next to this notebook

OPENALEX_URL = "https://api.openalex.org/works"
FIELDS = "id,doi,title,publication_year,cited_by_count,type,abstract_inverted_index,primary_location,authorships"


def openalex_search(query: str, since_year: int, n: int, max_retries: int = 25) -> list:
    params = {
        "search": query,
        "filter": f"from_publication_date:{since_year}-01-01,has_abstract:true,type:article|preprint|review",
        "per_page": min(n, 200),
        "select": FIELDS,
    }
    if os.environ.get("OPENALEX_API_KEY"):
        params["api_key"] = os.environ["OPENALEX_API_KEY"]
    for attempt in range(max_retries):
        r = requests.get(OPENALEX_URL, params=params, timeout=60)
        if r.status_code == 429 or r.status_code >= 500:
            try:
                wait = float(r.json().get("retryAfter") or r.headers.get("retry-after") or 5)
            except ValueError:
                wait = 5 * (attempt + 1)
            tqdm.write(f"  OpenAlex {r.status_code} on '{query}', retrying in {wait:.0f}s...")
            time.sleep(wait + 1)
            continue
        r.raise_for_status()
        return r.json()["results"]
    tqdm.write(f"  giving up on '{query}' after {max_retries} tries (set OPENALEX_API_KEY to avoid rate limits)")
    return []


papers: dict[str, Paper] = {}  # OpenAlex id -> Paper
for kind, qs in queries.items():
    for q in tqdm(qs, desc=kind):
        for w in openalex_search(q, SINCE_YEAR, PER_QUERY):
            if w["id"] not in papers:
                paper = Paper.from_openalex(w)
                if not paper.title or not paper.abstract:
                    continue
                papers[paper.id] = paper
            papers[w["id"]].kinds.add(kind)
            papers[w["id"]].matched_queries.add(q)

print(f"{len(papers)} unique candidate papers")

## 4. Embed with SPECTER2 and rank

SPECTER2 has two adapters. `adhoc_query` embeds short search text like your problem description, and `proximity` embeds papers. Paper embeddings are cached in `cache/`, so re-runs only embed new papers.

The first run downloads about 450 MB of model weights.

In [ ]:
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")
model.load_adapter("allenai/specter2", source="hf", load_as="proximity", set_active=False) # embedding papers
model.load_adapter("allenai/specter2_adhoc_query", source="hf", load_as="adhoc_query", set_active=False) # embedding short search text
model.eval()

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)


def embed(texts: list, adapter: str, batch_size: int = 16) -> np.ndarray:
    model.set_active_adapters(adapter)
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"embedding ({adapter})", disable=len(texts) <= batch_size):
        batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True, max_length=512,
                          return_tensors="pt", return_token_type_ids=False).to(device)
        with torch.no_grad():
            out.append(model(**batch).last_hidden_state[:, 0, :].cpu().numpy())  # [CLS] token
    return np.vstack(out)

In [ ]:
CACHE = Path("cache/paper_embeddings.npz")
CACHE.parent.mkdir(exist_ok=True)
cache = dict(np.load(CACHE)) if CACHE.exists() else {}

ids = list(papers)
todo = [i for i in ids if i.split("/")[-1] not in cache]
if todo:
    texts = [papers[i].title + tokenizer.sep_token + papers[i].abstract for i in todo]
    for i, v in zip(todo, embed(texts, "proximity")):
        cache[i.split("/")[-1]] = v
    np.savez(CACHE, **cache)
print(f"embedded {len(todo)} new papers ({len(ids) - len(todo)} from cache)")

paper_vecs = np.vstack([cache[i.split("/")[-1]] for i in ids])
query_vec = embed([" ".join(PROBLEM.split())], "adhoc_query")[0]


def normalize(m):
    return m / np.linalg.norm(m, axis=-1, keepdims=True)

similarity = normalize(paper_vecs) @ normalize(query_vec)

In [ ]:
import pandas as pd

df = pd.DataFrame([papers[i] for i in ids])
df["similarity"] = similarity

# SPECTER similarities sit in a narrow band, so rank on z-scores and add a mild citation bonus
# (log-scaled, and per year since publication so new papers aren't buried).
z = lambda s: (s - s.mean()) / (s.std() or 1)
age = (pd.Timestamp.now().year - df["year"]).clip(lower=0) + 1
df["score"] = z(df["similarity"]) + CITATION_WEIGHT * z(np.log1p(df["citations"] / age))
df["kind"] = df["kinds"].map(lambda k: " + ".join(sorted(k)))
df = df.sort_values("score", ascending=False).reset_index(drop=True)
df.index += 1

## 5. Results

In [ ]:
from IPython.display import HTML, display

top = df.head(TOP_K)
display(HTML(top.assign(
    title=lambda d: [f'<a href="{u}" target="_blank">{t}</a>' for u, t in zip(d["url"], d["title"])],
    similarity=lambda d: d["similarity"].round(3),
    score=lambda d: d["score"].round(2),
)[["title", "year", "venue", "citations", "kind", "similarity", "score"]].to_html(escape=False)))

Read the abstracts of the top few:

In [ ]:
for rank, p in df.head(5).iterrows():
    print(f"#{rank}  {p['title']}  ({p['year']}, {p['venue']}, {p['citations']} citations)")
    print(f"    {p['url']}   [{p['kind']}]")
    print(f"    {p['abstract'][:700]}{'...' if len(p['abstract']) > 700 else ''}\n")

Best papers of each kind. This is useful when the analogous papers crowd out the same-domain ones, or the other way around:

In [ ]:
for kind in queries:
    subset = df[df["kinds"].map(lambda k: kind in k)].head(10)
    print(f"--- {kind} ---")
    for rank, p in subset.iterrows():
        print(f"  #{rank:<4} {p['similarity']:.3f}  {p['title']} ({p['year']})")

## 6. Save

In [ ]:
Path("results").mkdir(exist_ok=True)
out = Path("results") / f"papers_{pd.Timestamp.now():%Y%m%d_%H%M}.csv"
df.drop(columns=["kinds", "matched_queries"]).assign(
    matched_queries=df["matched_queries"].map(lambda q: "; ".join(sorted(q)))
).to_csv(out)
print(f"saved {len(df)} ranked papers to {out}")